# 01 — Square-Law vs Hilbert Transform
# Giai đoạn 3 — Mục 3.1a — So sánh thời gian thực thi và RAM
**Đầu ra**: `outputs/tables/table_3a_dsp_comparison.csv`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import timeit

from common import io_utils, dsp, pipeline, config as cfg

In [ ]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(TABLES_DIR / "manifest_filtered.csv")
BAND_HZ = (3200, 3800)

- Chọn tín hiệu mẫu

In [ ]:
sample_signal = io_utils.load_de_signal(Path(pipeline.pick_file(manifest, label='OR', load_hp=0)))

# Đo thời gian Square-Law

In [ ]:
def square_law_wrapper():
    return dsp.square_law_envelope(sample_signal, fs=12000, band=BAND_HZ, lp_cutoff=500)

time_square = timeit.timeit(square_law_wrapper, number=100) / 100 * 1000

# Đo thời gian Hilbert

In [ ]:
def hilbert_wrapper():
    return dsp.hilbert_envelope(sample_signal, fs=12000, band=BAND_HZ)

time_hilbert = timeit.timeit(hilbert_wrapper, number=100) / 100 * 1000

# RAM ước lượng (buffer size)

In [ ]:
ram_square = len(sample_signal) * 4  # float32
ram_hilbert = len(sample_signal) * 8  # complex128

table_3a = pd.DataFrame({
    'Method': ['Square-Law', 'Hilbert'],
    'Latency (ms)': [time_square, time_hilbert],
    'RAM (bytes)': [ram_square, ram_hilbert]
})
table_3a.to_csv(TABLES_DIR / "table_3a_dsp_comparison.csv", index=False)
table_3a